In [8]:
# Author: Om Mohite

import os
import pandas as pd
from sqlalchemy import URL, create_engine, text
from dotenv import load_dotenv

load_dotenv()

db_password = os.getenv("DB_PASSWORD")
if not db_password:
    raise ValueError("Set DB_PASSWORD in your .env file.")

server_url = URL.create(
    drivername="mysql+pymysql",
    username=os.getenv("DB_USER", "root"),
    password=db_password,
    host=os.getenv("DB_HOST", "localhost"),
    port=int(os.getenv("DB_PORT", "3306")),
)

server_engine = create_engine(server_url, pool_pre_ping=True)

with server_engine.begin() as conn:
    conn.execute(text("CREATE DATABASE IF NOT EXISTS test_case_16"))

server_engine.dispose()

connection_url = URL.create(
    drivername="mysql+pymysql",
    username=os.getenv("DB_USER", "root"),
    password=db_password,
    host=os.getenv("DB_HOST", "localhost"),
    port=int(os.getenv("DB_PORT", "3306")),
    database="test_case_16",
)

engine = create_engine(connection_url, pool_pre_ping=True)

print("Database test_case_16 is ready.")

Database test_case_16 is ready.


In [9]:
# Author: Om Mohite

import re
from pathlib import Path

sql_file = Path.cwd() / "test_case_16.sql"
if not sql_file.exists():
    raise FileNotFoundError(f"Place test_case_16.sql beside the notebook: {Path.cwd()}")

sql_script = sql_file.read_text(encoding="utf-8")
load_statements = []

for statement in sql_script.split(";"):
    statement = statement.strip()

    if re.match(
        r"^(CREATE TABLE|INSERT INTO sales\b)",
        statement,
        re.IGNORECASE,
    ):
        load_statements.append(statement)

with engine.begin() as conn:
    conn.exec_driver_sql("DROP TABLE IF EXISTS data_quality_issues")
    conn.exec_driver_sql("DROP TABLE IF EXISTS sales")

    for statement in load_statements:
        conn.exec_driver_sql(statement)

with engine.connect() as conn:
    total_rows = conn.execute(
        text("SELECT COUNT(*) FROM sales")
    ).scalar_one()

    unique_sale_ids = conn.execute(
        text("SELECT COUNT(DISTINCT sale_id) FROM sales")
    ).scalar_one()

    issue_table_exists = conn.execute(
        text("SHOW TABLES LIKE 'data_quality_issues'")
    ).first()

print("SQL file used:", sql_file.name)
print("Sales rows:", total_rows)
print("Distinct sale IDs:", unique_sale_ids)
print("Issue table ready:", issue_table_exists is not None)

SQL file used: test_case_16.sql
Sales rows: 18
Distinct sale IDs: 17
Issue table ready: True


In [10]:
# Author: Om Mohite

with engine.connect() as conn:
    sales_df = pd.read_sql(
        text("SELECT * FROM sales ORDER BY sale_id"),
        conn,
    )

print("Rows extracted:", len(sales_df))
print("Missing values by column:")
display(sales_df.isna().sum().to_frame("missing_count"))

display(sales_df)

Rows extracted: 18
Missing values by column:


,missing_count
sale_id,0
customer_id,2
product_id,0
order_date,2
quantity,1
unit_price,1
total_amount,1


,sale_id,customer_id,product_id,order_date,quantity,unit_price,total_amount
0,101,1.0,501,2026-11-01,2.0,500.0,1000.0
1,102,2.0,502,2026-11-02,1.0,250.0,250.0
2,103,3.0,503,2026-11-03,5.0,100.0,500.0
3,104,4.0,504,2026-11-04,3.0,200.0,600.0
4,105,5.0,505,2026-11-05,4.0,150.0,600.0
5,106,NaN,506,2026-11-06,2.0,300.0,600.0
6,107,7.0,507,None,3.0,100.0,300.0
7,108,8.0,508,2026-11-08,NaN,100.0,300.0
8,109,9.0,509,2026-11-09,2.0,NaN,200.0
9,110,10.0,510,2026-11-10,2.0,100.0,NaN


In [11]:
# Author: Om Mohite

rules = {
    "missing_customer_id": {
        "query": """
            SELECT COUNT(*)
            FROM sales
            WHERE customer_id IS NULL
        """,
        "severity": "HIGH",
    },
    "missing_order_date": {
        "query": """
            SELECT COUNT(*)
            FROM sales
            WHERE order_date IS NULL
        """,
        "severity": "HIGH",
    },
    "duplicate_sale_id": {
        "query": """
            SELECT COUNT(*)
            FROM (
                SELECT sale_id
                FROM sales
                GROUP BY sale_id
                HAVING COUNT(*) > 1
            ) AS duplicate_groups
        """,
        "severity": "HIGH",
    },
    "invalid_quantity": {
        "query": """
            SELECT COUNT(*)
            FROM sales
            WHERE quantity IS NULL OR quantity <= 0
        """,
        "severity": "HIGH",
    },
    "negative_price": {
        "query": """
            SELECT COUNT(*)
            FROM sales
            WHERE unit_price < 0
        """,
        "severity": "MEDIUM",
    },
    "inconsistent_total_amount": {
        "query": """
            SELECT COUNT(*)
            FROM sales
            WHERE total_amount IS NULL
               OR quantity IS NULL
               OR unit_price IS NULL
               OR ABS(total_amount - (quantity * unit_price)) > 0.01
        """,
        "severity": "HIGH",
    },
}

print("Quality rules defined:", len(rules))

Quality rules defined: 6


In [12]:
# Author: Om Mohite

def data_quality_checks(engine, rules):
    issues = []

    with engine.connect() as conn:
        for rule_name, rule_details in rules.items():
            failed_records = conn.execute(
                text(rule_details["query"])
            ).scalar_one()

            if failed_records > 0:
                issues.append({
                    "pipeline_name": "sales_pipeline",
                    "table_name": "sales",
                    "rule_name": rule_name,
                    "severity": rule_details["severity"],
                    "failed_records": int(failed_records),
                })

    return issues

issues = data_quality_checks(engine, rules)
issues_df = pd.DataFrame(issues)

print("Quality rules with failures:", len(issues))
display(issues_df)

Quality rules with failures: 6


,pipeline_name,table_name,rule_name,severity,failed_records
0,sales_pipeline,sales,missing_customer_id,HIGH,2
1,sales_pipeline,sales,missing_order_date,HIGH,2
2,sales_pipeline,sales,duplicate_sale_id,HIGH,1
3,sales_pipeline,sales,invalid_quantity,HIGH,4
4,sales_pipeline,sales,negative_price,MEDIUM,2
5,sales_pipeline,sales,inconsistent_total_amount,HIGH,6


In [13]:
# Author: Om Mohite

with engine.begin() as conn:
    conn.execute(text("DELETE FROM data_quality_issues"))

    if issues:
        conn.execute(
            text("""
                INSERT INTO data_quality_issues
                    (pipeline_name, table_name, rule_name, severity, failed_records)
                VALUES
                    (:pipeline_name, :table_name, :rule_name, :severity, :failed_records)
            """),
            issues,
        )

with engine.connect() as conn:
    saved_issues = pd.read_sql(
        text("""
            SELECT
                pipeline_name,
                table_name,
                rule_name,
                severity,
                failed_records,
                status
            FROM data_quality_issues
            ORDER BY issue_id
        """),
        conn,
    )

print("Issue records saved:", len(saved_issues))
display(saved_issues)

Issue records saved: 6


,pipeline_name,table_name,rule_name,severity,failed_records,status
0,sales_pipeline,sales,missing_customer_id,HIGH,2,OPEN
1,sales_pipeline,sales,missing_order_date,HIGH,2,OPEN
2,sales_pipeline,sales,duplicate_sale_id,HIGH,1,OPEN
3,sales_pipeline,sales,invalid_quantity,HIGH,4,OPEN
4,sales_pipeline,sales,negative_price,MEDIUM,2,OPEN
5,sales_pipeline,sales,inconsistent_total_amount,HIGH,6,OPEN
